# 07 — Transfer Learning and Fine-Tuning

This notebook reuses a network pretrained on ImageNet (e.g. ResNet-18/50, EfficientNet or MobileNet from `torchvision`) to classify H&M garments. Two strategies are compared: **feature extraction**, where the pretrained backbone is frozen and only a new classification head is trained, and **fine-tuning**, where some or all backbone layers are unfrozen and trained with a small learning rate.

It addresses the course's "Transfer Learning and Fine Tuning" learning strategy and the practical pitfalls discussed in class, such as the correct handling of batch-normalisation statistics when freezing layers and the importance of using the backbone's own preprocessing.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Vía B · Lo habitual en aplicación: reutilizar representaciones

Interpretamos esta pista de clase como la posibilidad de resolver el problema reutilizando un modelo, frente a entrenarlo todo desde cero. Es una práctica habitual, no una obligación universal ni una garantía de mejores resultados. Primero: encoder congelado + cabeza entrenable. Después: fine-tuning controlado. Comparar con scratch manteniendo los grupos de producto y explicitando diferencias de preprocesamiento y presupuesto.

**Preentrenamiento y autosupervisión no son sinónimos.** Un checkpoint convencional de clasificación ImageNet puede estar preentrenado con etiquetas humanas; no se etiquetará como self-supervised. Una alternativa autosupervisada aprende representaciones con tareas construidas a partir de los propios datos; [SimCLR](https://arxiv.org/abs/2002.05709) proporciona un ejemplo contrastivo visual. Si se prueba esa alternativa, identificar checkpoint, datos, objetivo y protocolo original, sin suponer que deba reproducirse su costoso preentrenamiento aquí.

Los dos tipos de encoder pueden producir embeddings visuales para una cabeza clasificadora. Esos vectores no son word vectors ni generación de imágenes. Esta vía pretende explicar transferencia, congelación y adaptación; el rendimiento es evidencia que interpretar, no el único objetivo.


## Protocolo revisado antes del entrenamiento · 20/11/2026

La referencia común es `config/experimental_protocol_v1.json`. Es un diseño fijado antes de entrenar las redes, no una lista de resultados obtenidos. La versión equilibrada de 18.000 fotos se conserva como experimento exploratorio. Sus métricas describen las doce clases seleccionadas con ese muestreo; no estiman automáticamente el comportamiento sobre el catálogo original.

**Por defecto:** entropía cruzada sin pesos, sin aumentos y con normalización de train a la resolución de entrada. Pesos de clase y aumentos se activarán como ablaciones explícitas. Los pesos guardados en 02 son una opción calculada, no una instrucción de aplicarlos siempre. Cada experimento registrará resolución, preprocesamiento, IDs de productos, configuración, semilla y presupuesto.

**Métricas fijadas:** macro-F1 primaria; accuracy secundaria; precisión, recall, F1 y soporte por clase; matriz de confusión en recuentos y normalizada por filas. Comparar parámetros entrenables, tiempo de entrenamiento, latencia por imagen, memoria pico y hardware. Selección por macro-F1 de validation; early stopping por pérdida de validation; tres semillas (42, 123 y 2026). El test permanece cerrado hasta congelar decisiones; intervalos finales mediante remuestreo de grupos de producto, no imágenes independientes. Los baselines deterministas no necesitan simular tres entrenamientos diferentes.

### H3 · Aislar el efecto de los pesos preentrenados

Comparar la misma arquitectura de backbone, resolución y transformación de entrada con inicialización aleatoria y preentrenada. Publicar por separado encoder congelado + cabeza y fine-tuning: sus parámetros entrenables y presupuestos son distintos. Mantener train/validation, semilla, límite de épocas y criterio de selección. Informar coste local y reconocer el coste/datos externos del preentrenamiento. Una CNN pequeña a 64×64 frente a ResNet preentrenada a 224×224 es una comparación aplicada útil, pero no una estimación causal aislada del efecto del preentrenamiento.


## Objectives

- Load a pretrained backbone and replace its top classifier with a head for our garment classes.
- Train the head with a frozen backbone (feature extraction) and evaluate on validation.
- Fine-tune the last block(s) and then the whole network with a lower learning rate; compare both strategies.
- Handle batch normalisation correctly (inference mode / frozen running statistics) and explain why.
- Measure the effect of data augmentation and of training-set size (learning with fewer labelled images).
- Compare pretrained vs. from-scratch models (notebook 05) in accuracy, convergence speed and cost.

## Preview of the work

1. **Backbone selection.** candidate architectures, input size and normalisation expected by each one.
2. **Feature extraction.** frozen backbone + new Dense head.
3. **Fine-tuning.** progressive unfreezing, discriminative learning rates, early stopping.
4. **Batch-normalisation behaviour.** experiment showing the impact of BN mode when freezing.
5. **Data efficiency.** results with 10 %, 25 %, 50 % and 100 % of the training data.
6. **Embeddings.** t-SNE / UMAP of backbone features coloured by class.
7. **Validation results and conclusions.** best transfer-learning configuration.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders from notebook 02 (224×224 variant), from-scratch results from notebook 05 |
| **Outputs** | Metrics, checkpoints and embedding figures in `results/` |
| **Course material** | `MLLB_TransferLearning_Image_and-Text.pptx`, `MLLB_Keras_transfer_learning_v2.ipynb`, `pytorch_transfer_learning_colab.ipynb` |